Running this notebook requires that a kafka serve is running on the local system:

The first time you run kafka you can just call:
```
docker run -d --name kafka -p 9092:9092 apache/kafka:3.7.0
```

After that, you can run this to clean up and restart from the CLI. Or delete the 
container using the Docker Desktop app.
```
docker kill kafka; docker rm kafka;  docker run -d --name kafka -p 9092:9092 apache/kafka:3.7.0
```
The first two command (kill and rm) are just to clean up any existing kafka container, and the third command starts a new kafka server in a docker container.

Then run the `producer.py` script in this same directory to produce randomly sized batches of 
messages with random delays in between batches.

Once both are running, you can run this notebook. 

Re-running the producer.py script while this notebook is running will cause new batches to 
be produced and consumed by the notebook, demonstrating that the notebook can handle batches
of varying sizes and delays and will wait for long periods of time between batches.

Restarting this notebook doesn't seem to pick up new messages off the stream though.
It seems to get stuck "Waiting for samples".

In [ ]:
from hyrax import Hyrax

h = Hyrax()

h.set_config("model.name", "HyraxLoopback")
h.set_config("infer_stream.model_weights_file", "foo.bar")
h.set_config("infer_stream.save_model_output", True)

In [ ]:
data_request = {
    "infer_stream": {
        "data": {
            "dataset_class": "KafkaStreamDataset",
            "data_location": "kafka://localhost:9092/my-topic",
            "primary_id_field": "id",
            "fields": ["image", "label"],
        }
    }
}
h.set_config("data_request", data_request, over_write=True)

In [ ]:
ds = h.prepare()

In [ ]:
print("Setting up context manager")
with h.infer_stream() as session:
    print("Waiting for samples")

    # This for loop iterates over the streaming dataset
    for i, batch in enumerate(session.data_loader):
        result = session.process(batch)
        print(f"Processed batch {i} with {len(batch['object_id'])} samples.")

I think that the follow is what the `process.py` main function loop will look like in `hyrax_alerts`.

In [ ]:
# with h.infer_stream() as session:
#     for batch in session.data_loader:
#         batch = session.data_loader.pre_filter(batch)
#         batch = session.data_loader.pre_process(batch)
#         if batch:
#             result = session.process(batch)

#         result = e.post_process(result)
#         for e in emitter_list:
#             e_result = e.post_filter(result)
#             e.emit(e_result)